# Contact extraction

Turns each climbing video into a hand-hold sequence: an ordered list of which hold each hand grabbed and when. This is what the next-hold model trains on. We also score the list against the hand-labeled ground truths and adjust the rules as nedded.

## Cell 1: Setup

Settings:
- `FPS`: every video was recorded at 30 frames per second; pose CSVs give frame numbers, and dividing by `FPS` turns them into seconds.
- `CONF_MIN`: wrist confidence below this is treated as "hand not seen". 0.5 from pose-estimation notebook.
- `MAX_GAP_S`: hidden stretches up to this long are filled by drawing a straight line between the points either side. Longer ones are left hidden.
- `DIST_FACTOR`: how far the hand point may be from a hold's outline and still count as on it.
- `DWELL_S`: how long the hand must stay on a hold to count as a grab.
- `VOLUME_DWELL_S`: the same for volumes.
- `START_WINDOW_S`: a hand's first contact counts as a start hold if it begins before `Climb_Start` + this value.
- `TRIM_PAD_S`: how much video to keep before `Climb_Start` and after `Climb_End`. A little before the start is needed so the hands are already on the start holds when the clip begins.
- `INCLUDE_VOLUMES`: volumes count as on every route at their wall, as decided.
- `DROP_FALL_LAST` / `FALL_LAST_MAX_S`: on falls, drop the last contact if it lasted less than this.
- `MATCH_WINDOW_S`: scoring tolerance. A predicted grab matches a labeled one if it's the same video, hand and hold and within 1 s.

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import cv2
from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ECS-174-Project'
MAP_DIR = f'{BASE}/hold-maps'
POSE_DIR = f'{BASE}/annotated-csv'
GT_PATH = f'{BASE}/ground-truth/ground_truth.csv'
OUT_PATH = f'{BASE}/sequences.csv'
CHECK_DIR = f'{BASE}/contact-checks'
os.makedirs(CHECK_DIR, exist_ok=True)

EXCLUDE_POSITIONS = {'pos2', 'pos4', 'pos6', 'pos7', 'pos11', 'pos18', 'pos22', 'pos25'}
EXCLUDE_VIDEOS = {'ronish-blue-14'}

FPS = 30.0              # all videos are 30 fps
CONF_MIN = 0.5
MAX_GAP_S = 0.5
DIST_FACTOR = 1.0
DWELL_S = 0.3
VOLUME_DWELL_S = 1.0    # longer than holds 
END_DWELL_S = 0.1
SWITCH_MARGIN = 0.5     # stay on the current hold unless another is closer by this many hold widths
RETURN_MAX_S = 5.0      # hand leaves a hold and comes back within this time means what it touched between was a tap
INFERRED_AT = 'mid'     # grab seen only after a hidden stretch: 'mid' = middle of the stretch
START_WINDOW_S = 1.0
TRIM_PAD_S = 1.0
INCLUDE_VOLUMES = True
DROP_FALL_LAST = True
FALL_LAST_MAX_S = 1.0
MATCH_WINDOW_S = 1.0
START_ANY_TIME = True
SCORE_HIDDEN_INTERVAL = True  # scoring: a grab made while the hand was hidden matches if the label falls in that hidden stretch (+/- 1 s)

## Cell 2: Load metadata and pick the videos

Reads `metadata.csv`, works out each video's name, position and route ID, and drops excluded positions and videos. The route ID comes from the filename the same way the redo cell does it. All route IDs are normalized.

In [ ]:
def video_name(v):
    """Filename without its video extension (keeps '.2' on later attempts)."""
    return re.sub(r'\.(mp4|mov|m4v|avi)$', '', str(v).strip(), flags=re.IGNORECASE)

def normalize_pos(s):
    m = re.search(r'pos0*(\d+)', str(s).lower())
    return f'pos{m.group(1)}' if m else None

def norm_route(r):
    return re.sub(r'[^a-z0-9]', '', str(r).lower())

def route_from_video(v):
    p = video_name(v).lower().split('-')
    return norm_route(f"{p[-2]}{p[-1].split('.')[0]}")

meta = pd.read_csv(f'{BASE}/metadata.csv')
meta = meta[meta['Filename'].notna() & (meta['Filename'].astype(str).str.strip() != '')].copy()
meta['video'] = meta['Filename'].map(video_name)
meta['key'] = meta['video'].str.lower()
meta['position_id'] = meta['Position_ID'].map(normalize_pos)
meta['route_id'] = meta['Filename'].map(route_from_video)
for c in ['Climb_Start', 'Climb_End']:
    meta[c] = pd.to_numeric(meta[c], errors='coerce')

reason = pd.Series('', index=meta.index)
reason[meta['position_id'].isna()] = 'no position'
reason[meta['position_id'].isin(EXCLUDE_POSITIONS)] = 'excluded position'
reason[meta['key'].isin(EXCLUDE_VIDEOS)] = 'excluded video'
reason[meta['key'].duplicated(keep='first')] = 'duplicate row'
skipped = meta[reason != ''].assign(reason=reason[reason != ''])
videos = meta[reason == ''].reset_index(drop=True)

print(f'{len(videos)} videos kept, {videos.route_id.nunique()} routes, '
      f'{videos.groupby(["route_id", "position_id"]).ngroups} route/position pairs')
print(f'{len(skipped)} skipped:', skipped.groupby('reason').size().to_dict())
print('Missing Climb_Start or Climb_End (whole video used):',
      videos.loc[videos.Climb_Start.isna() | videos.Climb_End.isna(), 'video'].tolist())

FALL_WORDS = ('fall', 'fell', 'flop', 'drop', 'fail')
print('\nOutcome values:', videos['Outcome'].astype(str).str.strip().str.lower().value_counts().to_dict())
videos['fell'] = videos['Outcome'].astype(str).str.lower().str.contains('|'.join(FALL_WORDS))
print('Counted as falls:', int(videos.fell.sum()))

## Cell 3: Load hold maps and routes, build each route's candidate holds

Reads the holds, their outlines and `routes.csv`, then makes the list of candidate holds for every route/position pair: the `on_route=1` holds plus every volume at that position.

Also works out each position's hold width which every distance setting is multiplied by.

In [ ]:
holds = pd.read_csv(f'{MAP_DIR}/holds.csv')
holds['position_id'] = holds['position_id'].map(normalize_pos)
holds['hold_id'] = holds['hold_id'].astype(int)
holds['cls'] = holds['cls'].astype(str).str.lower()

with open(f'{MAP_DIR}/hold_polygons.json') as f:
    raw = json.load(f)
polys = {}
for p, d in raw.items():
    polys[normalize_pos(p)] = {int(h): [np.asarray(part, np.float32).reshape(-1, 1, 2)
                                        for part in parts if len(part) >= 3]
                               for h, parts in d.items()}

hw = holds[holds.cls == 'hold'].groupby('position_id')['width'].median()
hold_width = holds.groupby('position_id')['width'].median()
hold_width.update(hw)
print(f'{len(holds)} holds at {holds.position_id.nunique()} positions; '
      f'hold width {hold_width.min():.0f}-{hold_width.max():.0f} px')

routes = pd.read_csv(f'{MAP_DIR}/routes.csv')
routes = routes.loc[:, ~routes.columns.astype(str).str.startswith('Unnamed')]
routes['position_id'] = routes['position_id'].map(normalize_pos)
routes['route_id'] = routes['route_id'].map(norm_route)
routes['hold_id'] = routes['hold_id'].astype(int)
routes['on_route'] = pd.to_numeric(routes['on_route'], errors='coerce').fillna(0).astype(int)
routes['role'] = routes['role'].fillna('').astype(str).str.lower()

hold_info = holds.set_index(['position_id', 'hold_id'])
candidates = {}   # (route_id, position_id) -> DataFrame of candidate holds
problems = []
for (r, p), g in videos.groupby(['route_id', 'position_id']):
    rr = routes[(routes.route_id == r) & (routes.position_id == p)]
    on = rr[rr.on_route == 1][['hold_id', 'role']].copy()
    if len(rr) == 0:
        problems.append((r, p, 'no rows in routes.csv'))
    elif not on.role.str.contains('start').any():
        problems.append((r, p, 'no start hold'))
    elif not on.role.str.contains('end').any():
        problems.append((r, p, 'no end hold'))
    if INCLUDE_VOLUMES:
        vol = holds[(holds.position_id == p) & (holds.cls == 'volume')][['hold_id']].assign(role='')
        on = pd.concat([on, vol[~vol.hold_id.isin(on.hold_id)]])
    keep = []
    for h in on.hold_id:
        if (p, h) not in hold_info.index or not polys.get(p, {}).get(h):
            problems.append((r, p, f'hold {h} has no outline'))
            keep.append(False)
        else:
            keep.append(True)
    on = on[keep].copy()
    on['area'] = [hold_info.loc[(p, h), 'area'] for h in on.hold_id]
    on['cls'] = [hold_info.loc[(p, h), 'cls'] for h in on.hold_id]
    candidates[(r, p)] = on.reset_index(drop=True)

no_cands = [k for k, v in candidates.items() if len(v) == 0]
print(f'{len(candidates)} route/position pairs; median {np.median([len(v) for v in candidates.values()]):.0f} '
      f'candidate holds each; {len(no_cands)} with no candidate holds')
for r, p, msg in problems:
    print(f'  PROBLEM {r} {p}: {msg}')

## Cell 4: Load and clean the hand tracks

For every video and hand, turn the pose CSV into a clean track of (time, x, y).
1. Frame number ÷ `FPS` → seconds
2. A point counts as hidden if no person was found or its wrist confidence is below `CONF_MIN`
3. Hidden stretches no longer than `MAX_GAP_S` are filled in with a straight line. Longer ones stay hidden.
4. The track is cut to `Climb_Start − TRIM_PAD_S` … `Climb_End + TRIM_PAD_S`, so we don't count grabs that happen before this point.

In [ ]:
pose_files = {}   # video key -> pose CSV path
for f in sorted(os.listdir(POSE_DIR)):
    m = re.match(r'(.+?)(_hands)?\.csv$', f, flags=re.IGNORECASE)
    if m:
        k = video_name(m.group(1)).lower()
        if m.group(2) or k not in pose_files:
            pose_files[k] = os.path.join(POSE_DIR, f)

def fill_short_gaps(t, x, y, max_gap):
    """Straight-line fill of hidden stretches no longer than max_gap seconds."""
    x, y = x.copy(), y.copy()
    idx = np.flatnonzero(~np.isnan(x))
    for a, b in zip(idx[:-1], idx[1:]):
        if b - a > 1 and t[b] - t[a] <= max_gap:
            w = (t[a + 1:b] - t[a]) / (t[b] - t[a])
            x[a + 1:b] = x[a] + w * (x[b] - x[a])
            y[a + 1:b] = y[a] + w * (y[b] - y[a])
    return x, y

def load_tracks(v):
    """Returns {'L': DataFrame(t, x, y), 'R': ...} in video pixels."""
    df = pd.read_csv(pose_files[v.key]).sort_values('frame')
    t = df['frame'].to_numpy(float) / FPS
    lo = (v.Climb_Start - TRIM_PAD_S) if pd.notna(v.Climb_Start) else -np.inf
    hi = (v.Climb_End + TRIM_PAD_S) if pd.notna(v.Climb_End) else np.inf
    found = df['person_found'].fillna(0).astype(float).to_numpy() > 0
    out = {}
    for hand, px in (('L', 'l'), ('R', 'r')):
        x = df[f'{px}hand_x'].to_numpy(float)
        y = df[f'{px}hand_y'].to_numpy(float)
        conf = df[f'{px}wrist_conf'].fillna(0).to_numpy(float)
        bad = ~found | (conf < CONF_MIN) | np.isnan(x) | np.isnan(y)
        x, y = np.where(bad, np.nan, x), np.where(bad, np.nan, y)
        xr, yr = fill_short_gaps(t, x, y, MAX_GAP_S)
        keep = (t >= lo) & (t <= hi)
        out[hand] = pd.DataFrame({'t': t[keep], 'x': xr[keep], 'y': yr[keep]})
    return out

tracks, summary = {}, []
no_pose = [k for k in videos.key if k not in pose_files]
for v in videos.itertuples():
    if v.key not in pose_files:
        continue
    tr = load_tracks(v)
    tracks[v.key] = tr
    summary.append({'video': v.video, 'samples': len(tr['L']),
                    'L_visible': tr['L'].x.notna().mean(), 'R_visible': tr['R'].x.notna().mean()})
summary = pd.DataFrame(summary)
print(f'{len(tracks)} videos loaded; {len(no_pose)} with no pose CSV:', no_pose)
print(f'Median visible share: L {summary.L_visible.median():.0%}, R {summary.R_visible.median():.0%}')
print('Lowest visible share:')
print(summary.assign(worst=summary[['L_visible', 'R_visible']].min(axis=1))
      .nsmallest(5, 'worst')[['video', 'samples', 'L_visible', 'R_visible']].to_string(index=False))

## Cell 5: Measure hand-to-hold distances

For every sample and every candidate hold, measures how far the hand point is from the hold's outline: 0 if the point is, otherwise the distance to the nearest edge.

In [ ]:
def outline_distance(pt, parts):
    """0 inside the outline, otherwise pixels to the nearest edge (over all parts)."""
    return max(0.0, min(-cv2.pointPolygonTest(c, pt, True) for c in parts))

dist = {}   # dist[key] = {'cands': DataFrame, 'width': px, 'L': (t, D), 'R': (t, D)}
near = []
for v in videos.itertuples():
    if v.key not in tracks:
        continue
    cands = candidates[(v.route_id, v.position_id)]
    parts = [polys[v.position_id][h] for h in cands.hold_id]
    entry = {'cands': cands, 'width': float(hold_width[v.position_id])}
    for hand in ('L', 'R'):
        tr = tracks[v.key][hand]
        D = np.full((len(tr), len(cands)), np.nan)
        for i, (x, y) in enumerate(zip(tr.x.to_numpy(), tr.y.to_numpy())):
            if not np.isnan(x):
                D[i] = [outline_distance((float(x), float(y)), p) for p in parts]
        entry[hand] = (tr.t.to_numpy(), D)
        if len(cands):
            m = np.nanmin(np.where(np.isnan(D), np.inf, D), axis=1)
            m = m[np.isfinite(m)]
            if len(m):
                near.append(np.median(m) / entry['width'])
    dist[v.key] = entry
print(f'Distances for {len(dist)} videos. Median nearest-hold distance per video/hand '
      f'(hold widths): {np.median(near):.2f}; 90th percentile {np.percentile(near, 90):.2f}')

## Cell 6: Contact rules

Functions that turn distances into contacts.

**`assign`**, per sample: which hold is the hand on?
- Hidden sample is hidden.
- Point inside one or more outlines then pick the smallest of them (ex. hold inside volume).
- Otherwise do the nearest hold, if it's within `DIST_FACTOR` hold widths otherwise no hold.
- A volume only wins if no ordinary hold is within range.

**`contacts`**, per hand: group samples into stretches on the same hold.
- Hidden samples don't break a run, ex. a hand on hold A, hidden, then on A again is one contact.
- A hand on A, hidden for a while, then on B gives a grab of B, marked `inferred=1`.
- Runs shorter than `DWELL_S` are dropped (ex. hand passing over a hold).
- Two runs on the same hold with nothing kept in between are merged (ex. readjustment)

**`label_and_order`**, both hands together:
- A hand's first contact that begins before `Climb_Start + START_WINDOW_S` is a `start`.
- A contact onto the hold the other hand is on at that moment is a `match`.
- Everything else is a `grab`.
- On a fall, a last contact shorter than `FALL_LAST_MAX_S` is dropped.
- Contacts are numbered in time order (`order`). `time_s` is when the hand arrived, in seconds from the start of the video, like the ground truth.

In [ ]:
HIDDEN, NONE = -2, -1

def assign(D, cands, thr, margin):
    """Per sample: index into cands of the hold the hand is on, NONE or HIDDEN.
    Sticky: once on a hold, the hand stays on it while it's within range, unless another
    hold is clearly closer or is a smaller hold the point is inside."""
    n = D.shape[0]
    lab = np.full(n, NONE)
    if D.shape[1] == 0:
        return lab                             # no candidate holds: never on a hold
    hidden = np.isnan(D).all(axis=1)
    lab[hidden] = HIDDEN
    area = cands['area'].to_numpy(float)
    is_vol = (cands['cls'] == 'volume').to_numpy()
    Dv = np.where(np.isnan(D), np.inf, D)
    cur = NONE
    for i in np.flatnonzero(~hidden):
        d = Dv[i]
        ok = d <= thr
        j = NONE
        if ok.any():
            if (ok & ~is_vol).any():
                ok &= ~is_vol                  # ordinary holds beat volumes
            idx = np.flatnonzero(ok)
            inside = idx[d[idx] <= 0]
            j = inside[np.argmin(area[inside])] if len(inside) else idx[np.argmin(d[idx])]
        if cur >= 0 and j != cur and d[cur] <= thr and not is_vol[cur]:
            clearly_closer = j >= 0 and d[j] + margin < d[cur]
            smaller_inside = j >= 0 and d[j] <= 0 and area[j] < area[cur]
            if not (clearly_closer or smaller_inside):
                j = cur                        # hand hasn't clearly left: stay
        lab[i] = j
        cur = j
    return lab

def contacts(t, lab, dwell, is_end):
    """Runs of the same hold, hidden samples bridged. dwell[i] = min duration for cands row i;
    is_end[i] = True for end holds."""
    if len(t) == 0:
        return []
    dt = float(np.median(np.diff(t))) if len(t) > 1 else 0.1
    runs, cur, last_vis_t = [], None, None
    for ti, l in zip(t, lab):
        if l == HIDDEN:
            continue
        gap_start = last_vis_t if last_vis_t is not None and ti - last_vis_t > MAX_GAP_S + 1e-9 else None
        last_vis_t = ti
        if cur is not None and l == cur['lab']:
            cur['end'] = ti
            continue
        if cur is not None:
            runs.append(cur)
        cur = {'lab': l, 'start': ti, 'end': ti, 'inferred': int(gap_start is not None),
               'gap_start': gap_start}
    runs.append(cur)
    kept = [r for r in runs if r['lab'] >= 0 and r['end'] - r['start'] + dt >= dwell[r['lab']] - 1e-9]
    for r in kept:
        r['end'] += dt
    # merge back-to-back runs on the same hold, and drop excursions: if the hand leaves a hold
    # and comes back to it within RETURN_MAX_S, whatever it touched in between was a tap
    merged = []
    for r in kept:
        back = next((k for k in range(len(merged) - 1, -1, -1) if merged[k]['lab'] == r['lab']), None)
        tap = back is not None and r['start'] - merged[back]['end'] <= RETURN_MAX_S and \
              not any(is_end[m['lab']] for m in merged[back + 1:])
        if back is not None and (back == len(merged) - 1 or tap):
            merged[back]['end'] = r['end']
            del merged[back + 1:]
        else:
            merged.append(dict(r))
    for r in merged:                           # grab seen only after a hidden stretch: estimate its time
        r['seen'] = r['start']
        if r['inferred'] and INFERRED_AT == 'mid':
            r['start'] = (r['gap_start'] + r['start']) / 2
    return merged

def label_and_order(v, per_hand, cands):
    rows = []
    climb_start = v.Climb_Start if pd.notna(v.Climb_Start) else -np.inf
    for hand, cs in per_hand.items():
        other = per_hand['R' if hand == 'L' else 'L']
        for j, c in enumerate(cs):
            hid = int(cands.hold_id.iloc[c['lab']])
            if j == 0 and c['start'] <= climb_start + START_WINDOW_S:
                kind = 'start'
            elif any(o['lab'] == c['lab'] and o['start'] <= c['start'] < o['end'] for o in other):
                kind = 'match'
            else:
                kind = 'grab'
            rows.append({'video': v.video, 'time_s': round(c['start'], 2), 'end_s': round(c['end'], 2),
                         'hand': hand, 'hold_id': hid, 'kind': kind,
                         'role': cands.role.iloc[c['lab']], 'inferred': c['inferred'],
                         'hidden_from': round(c['gap_start'], 2) if c['inferred'] else np.nan,
                         'seen_s': round(c['seen'], 2)})
    seq = pd.DataFrame(rows, columns=['video', 'time_s', 'end_s', 'hand', 'hold_id', 'kind', 'role', 'inferred',
                                      'hidden_from', 'seen_s'])
    seq = seq.sort_values(['time_s', 'hand']).reset_index(drop=True)
    is_end = seq.role.isin(['end', 'finish'])
    if is_end.any():                                   # nothing counts after the last grab of the end hold
        seq = seq[seq.time_s <= seq.time_s[is_end].max()].reset_index(drop=True)
    if DROP_FALL_LAST and v.fell and len(seq):
        last = seq.iloc[-1]
        if last.end_s - last.time_s < FALL_LAST_MAX_S:
            seq = seq.iloc[:-1]
    seq.insert(1, 'order', np.arange(1, len(seq) + 1))
    return seq

def extract(v, dist_factor=None, dwell=None):
    dist_factor = DIST_FACTOR if dist_factor is None else dist_factor
    dwell = DWELL_S if dwell is None else dwell
    e = dist[v.key]
    thr = dist_factor * e['width']
    role = e['cands']['role'].astype(str)
    dwells = np.where(e['cands']['cls'] == 'volume', max(dwell, VOLUME_DWELL_S), dwell)
    dwells = np.where(role.isin(['end', 'finish']), min(dwell, END_DWELL_S), dwells)
    margin = SWITCH_MARGIN * e['width']
    is_end = role.isin(['end', 'finish']).to_numpy()
    per_hand = {h: contacts(e[h][0], assign(e[h][1], e['cands'], thr, margin), dwells, is_end)
                for h in ('L', 'R')}
    return label_and_order(v, per_hand, e['cands'])

## Cell 7: Extract videos and write `sequences.csv`

Runs the rules on every video and save one row per contact.

In [ ]:
seqs = []
for v in videos.itertuples():
    if v.key in dist:
        seqs.append(extract(v).assign(route_id=v.route_id, position_id=v.position_id))
sequences = pd.concat(seqs, ignore_index=True)
sequences.to_csv(OUT_PATH, index=False)

per = sequences.groupby('video').agg(n=('order', 'size'),
                                     first_start=('role', lambda r: 'start' in str(r.iloc[0])),
                                     last_finish=('role', lambda r: 'end' in str(r.iloc[-1])))
per = per.join(videos.set_index('video')[['fell']])
empty = sorted(set(videos.video[videos.key.isin(dist.keys())]) - set(per.index))
print(f'Wrote {len(sequences)} contacts for {len(per)} videos to {OUT_PATH}')
print(f'{len(empty)} videos with no contacts:', empty)
print(f'Contacts per video: median {per.n.median():.0f}, range {per.n.min()}-{per.n.max()}')
print(f'First contact on a start hold: {per.first_start.mean():.0%}')
print(f'Last contact on the finish hold: sends {per[~per.fell].last_finish.mean():.0%}, '
      f'falls {per[per.fell].last_finish.mean():.0%}')
print('Kinds:', sequences.kind.value_counts().to_dict(), '| inferred:', int(sequences.inferred.sum()))
print('\nSends that do not end on the finish:', per[~per.fell & ~per.last_finish].index.tolist())

## Cell 8: Compare to GT

Compares `sequences.csv` with `ground_truth.csv`. A predicted contact matches a labeled one if it's the same video, same hand, same hold, and within `MATCH_WINDOW_S` (1 s).

In [ ]:
def load_gt(path):
    g = pd.read_csv(path)
    g = g.loc[:, ~g.columns.astype(str).str.startswith('Unnamed')]
    g = g[g['video'].notna() & g['hold_id'].notna()].copy()
    g['video'] = g['video'].map(video_name)
    g['key'] = g['video'].str.lower()
    g['hold_id'] = pd.to_numeric(g['hold_id'], errors='coerce').astype('Int64')
    g['time_s'] = pd.to_numeric(g['time_s'], errors='coerce')
    h = g['hand'].astype(str).str.strip().str.lower().str[0].map({'l': 'L', 'r': 'R', 'b': 'B'})
    g['hand'] = h
    both = g[g.hand == 'B']                       # 'both' rows -> one row per hand
    g = pd.concat([g[g.hand != 'B'], both.assign(hand='L'), both.assign(hand='R')], ignore_index=True)
    g['kind'] = g['kind'].astype(str).str.strip().str.lower()
    u = g['unsure'] if 'unsure' in g.columns else pd.Series(0, index=g.index)
    g['unsure'] = u.astype(str).str.strip().str.lower().isin(['1', 'true', 'yes', 'y', 'x'])
    return g.dropna(subset=['hand', 'hold_id']).reset_index(drop=True)

gt = load_gt(GT_PATH)

def match_pairs(pred, lab, window=MATCH_WINDOW_S):
    """Greedy one-to-one matching by time. Returns list of (pred_index, label_index)."""
    pairs = []
    for li, l in lab.iterrows():
        c = pred[(pred.hand == l.hand) & (pred.hold_id == l.hold_id)]
        for pi, p in c.iterrows():
            dt = abs(p.time_s - l.time_s)
            if SCORE_HIDDEN_INTERVAL and p.get('inferred', 0) == 1 and pd.notna(p.get('hidden_from', np.nan)):
                dt = max(0.0, l.time_s - p.seen_s, p.hidden_from - l.time_s)   # 0 anywhere in the hidden stretch
            if dt <= window or (START_ANY_TIME and p.kind == 'start' and l.kind == 'start'):
                pairs.append((dt if dt <= window else window + dt, pi, li))
    used_p, used_l, out = set(), set(), []
    for _, pi, li in sorted(pairs):
        if pi not in used_p and li not in used_l:
            used_p.add(pi); used_l.add(li); out.append((pi, li))
    return out

def score(pred_all, lab_all, keys):
    rows, misses, kind_ok = [], [], []
    for k in keys:
        pred = pred_all[pred_all.key == k].reset_index(drop=True)
        lab = lab_all[lab_all.key == k].reset_index(drop=True)
        m = match_pairs(pred, lab)
        unsure_l = set(lab.index[lab.unsure])
        mp = {pi for pi, li in m if li not in unsure_l}
        ml = {li for pi, li in m if li not in unsure_l}
        neutral_p = {pi for pi, li in m if li in unsure_l}
        n_pred = len(pred) - len(neutral_p)
        n_lab = len(lab) - len(unsure_l)
        kind_ok += [pred.kind[pi] == lab.kind[li] for pi, li in m if li not in unsure_l]
        wrong = 0
        for li in set(lab.index) - ml - unsure_l:
            l = lab.loc[li]
            near = pred[(pred.hand == l.hand) & ((pred.time_s - l.time_s).abs() <= MATCH_WINDOW_S)]
            wrong += len(near) > 0
        rows.append({'video': k, 'labels': n_lab, 'predicted': n_pred, 'matched': len(mp),
                     'missed_wrong_hold': wrong, 'missed_not_found': n_lab - len(ml) - wrong,
                     'extra': n_pred - len(mp)})
    s = pd.DataFrame(rows, columns=['video', 'labels', 'predicted', 'matched', 'missed_wrong_hold',
                                    'missed_not_found', 'extra'])
    tot = s[['labels', 'predicted', 'matched']].sum()
    P = tot.matched / tot.predicted if tot.predicted else 0.0
    R = tot.matched / tot.labels if tot.labels else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    s['precision'] = (s.matched / s.predicted.replace(0, np.nan)).round(2)
    s['recall'] = (s.matched / s.labels.replace(0, np.nan)).round(2)
    return s, P, R, F, (np.mean(kind_ok) if kind_ok else np.nan)

seq_k = sequences.assign(key=sequences.video.str.lower())
gt_keys = sorted(set(gt.key) & set(dist.keys()))
print(f'{gt.key.nunique()} labeled videos; {len(gt_keys)} scored; '
      f'not in this run (excluded or missing):', sorted(set(gt.key) - set(gt_keys)))
print(f'{len(gt)} label rows, {int(gt.unsure.sum())} unsure (left out)')

scores, P, R, F, K = score(seq_k, gt, gt_keys)
scores.to_csv(f'{CHECK_DIR}/scores.csv', index=False)
print(f'\nOVERALL  precision {P:.2f}  recall {R:.2f}  F1 {F:.2f}  '
      f'(kind agrees on {K:.0%} of matches)')
print(f'Misses: {scores.missed_wrong_hold.sum()} wrong hold, {scores.missed_not_found.sum()} not found; '
      f'{scores.extra.sum()} extra predictions\n')
print(scores.sort_values('recall').to_string(index=False))

## Cell 9: Tune distance and dwell

Tryevery combination of `DIST_GRID` × `DWELL_GRID` and report F1.

In [ ]:
DIST_GRID = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
DWELL_GRID = [0.2, 0.3, 0.4, 0.6]

vid_by_key = {v.key: v for v in videos.itertuples()}
gt_routes = sorted({vid_by_key[k].route_id for k in gt_keys})
rng = np.random.default_rng(0)
half_a_routes = set(rng.permutation(gt_routes)[: (len(gt_routes) + 1) // 2])
keys_a = [k for k in gt_keys if vid_by_key[k].route_id in half_a_routes]
keys_b = [k for k in gt_keys if vid_by_key[k].route_id not in half_a_routes]
print(f'Half A: {len(keys_a)} videos ({len(half_a_routes)} routes); half B: {len(keys_b)} videos')
if not keys_b:
    print('Only one labeled route, so there is no held-out half.')

results = []
for df_ in DIST_GRID:
    for dw in DWELL_GRID:
        pred = pd.concat([extract(vid_by_key[k], df_, dw).assign(key=k) for k in gt_keys], ignore_index=True)
        _, Pa, Ra, Fa, _ = score(pred, gt, keys_a)
        _, Pb, Rb, Fb, _ = score(pred, gt, keys_b)
        _, Pall, Rall, Fall, _ = score(pred, gt, gt_keys)
        results.append({'dist_factor': df_, 'dwell_s': dw, 'F1_A': Fa, 'F1_B': Fb,
                        'P_all': Pall, 'R_all': Rall, 'F1_all': Fall})
res = pd.DataFrame(results).round(3)
res.to_csv(f'{CHECK_DIR}/sweep.csv', index=False)

print('\nF1 on half A (rows: DIST_FACTOR, columns: DWELL_S)')
print(res.pivot(index='dist_factor', columns='dwell_s', values='F1_A').to_string())
best = res.loc[res.F1_A.idxmax()]
cur = res[(res.dist_factor == DIST_FACTOR) & (res.dwell_s == DWELL_S)]
print(f'\nBest on A: DIST_FACTOR={best.dist_factor}, DWELL_S={best.dwell_s} '
      f'-> F1 A {best.F1_A:.2f}, held-out B {best.F1_B:.2f}')
if len(cur):
    c = cur.iloc[0]
    print(f'Current settings (cell 1): F1 A {c.F1_A:.2f}, held-out B {c.F1_B:.2f}')

## Cell 10: Score the unseen videos only

Score only the ground-truth videos that were not used while developing the contact rules.

In [ ]:
gt = load_gt(GT_PATH)                                   # reload: picks up the new rows
seq_k = sequences.assign(key=sequences.video.str.lower())
gt_keys = sorted(set(gt.key) & set(dist.keys()))

OLD = {'izzy-orange-2', 'izzy-yellow-9', 'ronish-yellow-16', 'izzy-blue-15', 'izzy-red-4',
       'izzy-green-5', 'ronish-green-5', 'ronish-blue-7', 'ronish-pink-5', 'ronish-teal-1'}
new_keys = [k for k in gt_keys if k not in OLD]
print('New labeled videos not scored (excluded position or missing):',
      sorted(set(gt.key) - set(dist.keys()) - OLD - {'clara-blue-4'}))
new_rows = gt[gt.key.isin(new_keys)]
print(f'{len(new_keys)} new videos, {len(new_rows)} label rows, {int(new_rows.unsure.sum())} unsure (left out)')

s_new, P, R, F, K = score(seq_k, gt, new_keys)
print(f'\nNEW VIDEOS ONLY: precision {P:.2f}  recall {R:.2f}  F1 {F:.2f}  (kind agrees on {K:.0%} of matches)')
print(f'Misses: {s_new.missed_wrong_hold.sum()} wrong hold, {s_new.missed_not_found.sum()} not found; '
      f'{s_new.extra.sum()} extra predictions\n')
print(s_new.sort_values('recall').to_string(index=False))